## Set Price regime

In [2]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


خواندن خروجی مرحله ی قبل

In [3]:
df = pd.read_feather("../Outputs/01_df.feather")

In [4]:
df["rent_mode"].unique()

[NaN, 'مقطوع', 'مجانی', 'توافقی']
Categories (3, object): ['توافقی', 'مجانی', 'مقطوع']

In [5]:
import numpy as np
import pandas as pd

# ۱. تعریف متغیرهای کمکی مالی (شرط‌های منطقی)
has_price = df["price_value"].notna() & (df["price_value"] > 0)
has_rent = df["rent_value"].notna() & (df["rent_value"] > 0)
has_credit = df["credit_value"].notna() & (df["credit_value"] > 0)

# بررسی حالت‌ها (Mode) و نادیده گرفتن مقادیر 'none' و 'مجانی'
mode_sell = df["price_mode"].notna() & (~df["price_mode"].isin(["none"]))
mode_rent = df["rent_mode"].notna() & (~df["rent_mode"].isin(["none", "مجانی"]))
mode_credit = df["credit_mode"].notna() & (~df["credit_mode"].isin(["none", "مجانی"]))

valid_sell_financials = has_price & mode_sell
valid_rent_financials = has_rent & mode_rent
valid_credit_financials = has_credit & mode_credit

# ۲. تعریف متغیرهای کمکی دسته‌بندی
is_sell_cat = df["cat2_slug"].str.contains("sell", na=False)
is_rent_cat = df["cat2_slug"].str.contains("rent", na=False) & (df["cat2_slug"] != "temporary-rent") # اجاره‌های سالانه
is_temp_rent = df["cat2_slug"] == "temporary-rent"
is_services = df["cat2_slug"] == "real-estate-services"

# ۳. اعمال منطق جدید با np.select
conditions = [
    # الف) دسته‌بندی فروش (Sell)
    is_sell_cat & valid_sell_financials,                        # فروش معتبر
    is_sell_cat & ~valid_sell_financials,                       # فروش نامعتبر (بدون قیمت)
    
    # ب) دسته‌بندی اجاره سالانه (Rent)
    is_rent_cat & ~valid_rent_financials & ~valid_credit_financials,  # اجاره نامعتبر (بدون رهن و اجاره)
    is_rent_cat & valid_credit_financials & ~valid_rent_financials,   # رهن کامل
    is_rent_cat & valid_credit_financials & valid_rent_financials,    # رهن و اجاره
    is_rent_cat & ~valid_credit_financials & valid_rent_financials,   # فقط اجاره
    
    # ج) سایر دسته‌ها
    is_temp_rent,
    is_services
]

choices = [
    "sell",
    "Invalid_sell",
    
    "Invalid_rent",
    "credit",
    "credit and rent",
    "rent",
    
    "temporary-rent",
    "services"
]

# اعمال شرایط روی دیتافریم
df["ad_type"] = np.select(conditions, choices, default="unknown")

# ۴. مشاهده خروجی و توزیع داده‌ها
print(df["ad_type"].value_counts(dropna=False))


ad_type
sell               565190
credit and rent    289100
credit              60121
Invalid_sell        32379
temporary-rent      29903
services            19403
rent                 2971
Invalid_rent          933
Name: count, dtype: int64


In [15]:
# invalid_sell = df[df["ad_type"] == "rent"]

invalid_sell= df[
    (df["ad_type"] == "credit") &
    (df["credit_mode"] != "مقطوع")
]
invalid_sell[
    [
        "cat2_slug",
        "title",
        # "price_mode",
        # "price_value",
        "rent_mode",
        "credit_mode",
        "rent_value",
        "credit_value",
        "description"
    ]
].head(20)


,cat2_slug,title,rent_mode,credit_mode,rent_value,credit_value,description


In [7]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

In [8]:
# invalid_sell = df[df["ad_type"] == "rent"]

invalid_sell= df[
    (df["ad_type"] == "credit") & 
    (df["credit_mode"] != "مقطوع")
]
invalid_sell[
    [
        "cat2_slug",
        "title",
        # "price_mode",
        # "price_value",
        "rent_mode",
        "credit_mode",
        "rent_value",
        "credit_value",
        "description"
    ]
].head(20)


,cat2_slug,title,rent_mode,credit_mode,rent_value,credit_value,description


In [ ]:
df["rent_mode"].unique()

[NaN, 'مقطوع', 'مجانی', 'توافقی']
Categories (3, object): ['توافقی', 'مجانی', 'مقطوع']

In [ ]:
df.loc[
    df["cat2_slug"].str.contains("rent", na=False),
    ["cat2_slug", "rent_mode", "credit_mode", "price_value", "rent_value", "credit_value","ad_type_by_cat","ad_type_by_price"]
]


,cat2_slug,rent_mode,credit_mode,price_value,rent_value,credit_value,ad_type_by_cat,ad_type_by_price
0,temporary-rent,NaN,NaN,NaN,NaN,NaN,rent,unknown
2,residential-rent,مقطوع,مقطوع,NaN,26000000.0,7.500000e+08,credit&rent,credit and rent
3,commercial-rent,مقطوع,مقطوع,NaN,95000000.0,9.500000e+08,credit&rent,credit and rent
5,residential-rent,مقطوع,مقطوع,NaN,6000000.0,2.500000e+08,credit&rent,credit and rent
6,commercial-rent,مقطوع,مقطوع,NaN,16000000.0,1.500000e+08,credit&rent,credit and rent
...,...,...,...,...,...,...,...,...
999987,residential-rent,مقطوع,مقطوع,NaN,3000000.0,3.000000e+07,credit&rent,credit and rent
999993,residential-rent,مقطوع,مقطوع,NaN,2000000.0,1.500000e+08,credit&rent,credit and rent
999996,residential-rent,مقطوع,مقطوع,NaN,45000000.0,1.000000e+09,credit&rent,credit and rent
999998,temporary-rent,NaN,NaN,NaN,NaN,NaN,rent,unknown
